# Thinking Outside the Grid: Squares in Squares 101

> **Uses the [Squares in Squares starter kit](https://www.kaggle.com/datasets/alexchilton/squares-in-squares-starter-kit)** (official metric, builders, certified best-known packings; 7,500+ downloads). If it saves you time, an upvote on the dataset keeps it maintained.

Put `n` unit squares into the smallest possible square box, without overlap. Squares may be **rotated**.
Do it for every `n` from 1 to 1,000.

This notebook explains the problem with pictures, shows the one trap that silently zeroes a submission,
and builds a submission that scores **22.72** (grid: 28.94), step by step. Everything is checked with the **official metric**, unchanged.

| section | what you get |
|---|---|
| 0 | the problem, the data, the file format |
| 1 | why tilting a square can ever help, derived for n = 5 |
| 2 | the score, and where its points are |
| 3 | how the scorer checks overlap, drawn |
| 4 | the float64 trap (it cost our first upload its only improvement) |
| 5 | a 40-line optimiser that rediscovers the tilt |
| 6 | staircases + a tilted block: one family, hundreds of n |
| 7 | the best-known table (n ≤ 324) |
| 8 | two free closures: add an L, remove squares |
| 9 | the submission: 28.94 → 22.72, step by step |
| 10 | where to go next: what the board did after this notebook, and where each idea is explained |

In [1]:
import sys, os, glob, math, time
# official metric (metric/squares-in-squares, unchanged), builder and toy optimiser: from the starter-kit dataset on Kaggle, src/ locally
SRC = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/build.py', recursive=True) + ['../src/build.py'] if os.path.exists(p))
sys.path.insert(0, SRC)
KAGGLE = os.path.exists('/kaggle/input')
DATA = next(d for d in ['/kaggle/input/competitions/squares-in-squares', '/kaggle/input/squares-in-squares', '../data']
            if os.path.exists(d + '/sample_submission.csv'))
OUT = '/kaggle/working' if KAGGLE else '../submissions'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, FancyArrowPatch
import metric, build, opt_demo
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('code from', SRC, '| data from', DATA)

## 0. The problem and the file

There is no training data and no hidden test set. The score is computed exactly from the coordinates you submit,
so the leaderboard number *is* the final number.

One row per unit square:

| column | meaning |
|---|---|
| `id` | row number 0, 1, 2, … (**required**: the metric drops it by name, a file without it scores nothing) |
| `n` | which puzzle, 1 … 1,000 |
| `square_id` | which square inside that puzzle, 0 … n−1 |
| `x`, `y` | centre of the square |
| `angle` | rotation in **radians** |

The box is not submitted. The scorer measures it: the side of the smallest axis-aligned square around all corners,
`L = max(max x − min x, max y − min y)`, so where you put the packing does not matter.
The sample submission is the plain grid:

In [2]:
sub = pd.read_csv(f'{DATA}/sample_submission.csv', dtype=str)
print(sub.shape, '=', sum(range(1, 1001)), 'squares in total')
sub[sub.n.isin(['3', '5'])]

In [3]:
def corners(x, y, a):
    c, s = math.cos(a), math.sin(a)
    return [(x + dx * c - dy * s, y + dx * s + dy * c) for dx, dy in [(.5, .5), (-.5, .5), (-.5, -.5), (.5, -.5)]]

def draw(ax, squares, title='', note=True):
    sq = [(float(x), float(y), float(a)) for x, y, a in squares]
    pts = [p for q in sq for p in corners(*q)]
    xs, ys = zip(*pts); L = max(max(xs) - min(xs), max(ys) - min(ys))
    ax.add_patch(Rectangle((min(xs), min(ys)), L, L, fill=False, lw=2.2, ec='crimson', zorder=3))
    for q in sq:
        tilted = abs(q[2]) > 1e-9
        ax.add_patch(Polygon(corners(*q), fc='#f4a63a' if tilted else '#4c7fb0', ec='#222', lw=.8, alpha=.9))
    n = len(sq)
    ax.set_title(title + (f'\nn={n}   L={L:.4f}   score={L / math.sqrt(n) - 1:.4f}' if note else ''), fontsize=9)
    ax.set_xlim(min(xs) - .25, min(xs) + L + .25); ax.set_ylim(min(ys) - .25, min(ys) + L + .25)
    ax.set_aspect('equal'); ax.axis('off')
    return L

fig, ax = plt.subplots(1, 4, figsize=(14, 3.9))
draw(ax[0], build.grid(4), 'perfect: 2 x 2')
draw(ax[1], build.grid(5), 'n=5 on the grid')
draw(ax[2], build.grid(10), 'n=10 on the grid')
draw(ax[3], build.five(), 'n=5 with one square tilted 45 degrees')
plt.suptitle('Red = the box the scorer measures.  Blue = upright squares, orange = rotated.', y=1.02)
plt.tight_layout(); plt.show()

One extra square beyond a perfect square costs a whole new row **and** column on the grid: n = 5 needs a 3 × 3 box
for 5 squares, n = 10 a 4 × 4 box for 10. Tilting fills the gaps the grid leaves.

## 1. Why tilting helps: n = 5 derived

Put four squares in the corners of a box of side `L`. They leave a cross-shaped gap.
A square turned 45° is a diamond whose half-diagonal is √2/2 ≈ 0.707: it is narrow at its tips, so its tips can
slide between two corner squares while its sides rest against their inner corners.

In [4]:
s2 = math.sqrt(2)
L5 = 2 + s2 / 2
fig, ax = plt.subplots(1, 2, figsize=(12, 5.2))
draw(ax[0], build.five(), 'the packing')
a = ax[1]; draw(a, build.five(), 'the geometry', note=False)
c = L5 / 2
a.plot([c, c + s2 / 2], [c, c], 'k--', lw=1); a.text(c + .1, c + .06, 'half-diagonal\n= sqrt(2)/2', fontsize=8)
a.plot([0, 1], [-.12, -.12], color='#4c7fb0', lw=3); a.text(.3, -.24, '1', ha='center')
a.plot([1, L5 - 1], [-.12, -.12], color='#f4a63a', lw=3); a.text(c, -.24, 'sqrt(2)/2', ha='center', fontsize=8)
a.plot([L5 - 1, L5], [-.12, -.12], color='#4c7fb0', lw=3); a.text(L5 - .5, -.24, '1', ha='center')
a.plot([1], [1], 'ko'); a.annotate('inner corner (1, 1)\ntouches the diamond side', (1, 1), (1.25, .45), fontsize=8,
                                   arrowprops=dict(arrowstyle='->'))
a.set_ylim(-.35, L5 + .25)
plt.tight_layout(); plt.show()
print(f'L = 1 + sqrt(2)/2 + 1 = {L5:.6f} instead of 3   ->   score {L5 / math.sqrt(5) - 1:.4f} instead of {3 / math.sqrt(5) - 1:.4f}')

The diamond's side is the line `x + y = c + √2/2` through the centre `(c, c)`, `c = L/2`; it touches the corner square
at (1, 1) when `2 = L/2 + √2/2`, i.e. `L = 2 + √2/2`. This is proven optimal (Göbel, 1979).

**Counterexample: n = 2.** Two unit squares need a box of side 2 however you turn them: each needs a full unit of width
in some direction, and a tilted square is *wider* (up to √2) than an upright one. Tilting only pays when it fills a gap
that upright squares cannot use. That is why the grid is optimal for every perfect square, and why all the action is
just above them.

## 2. The score

For each puzzle **score(n) = L / √n − 1**; the leaderboard is the sum over all 1,000 puzzles; **lower is better**.
√n is the side the box would need if no area at all were wasted, so the score is how much longer your side is than that
ideal, relative to it. A missing or invalid puzzle is charged the grid, `L = ⌈√n⌉`.

In [5]:
n = np.arange(1, 1001)
grid = np.ceil(np.sqrt(n))
g = grid / np.sqrt(n) - 1
fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
ax[0].plot(n, g, lw=.8, color='#4c7fb0'); ax[0].set_xscale('log')
ax[0].set_title('grid score per puzzle: a saw tooth'); ax[0].set_xlabel('n')
for k in [2, 3, 4, 5, 10, 20, 30]:
    ax[0].axvline(k * k, color='grey', lw=.4)
ax[1].plot(n, grid ** 2 - n, lw=.8, color='#c0504d'); ax[1].set_title('wasted area on the grid: L^2 - n'); ax[1].set_xlabel('n')
bins = [(1, 10), (11, 100), (101, 400), (401, 1000)]
tot = [g[a - 1:b].sum() for a, b in bins]
ax[2].bar([f'{a}-{b}' for a, b in bins], tot, color='#4c7fb0'); ax[2].set_title(f'grid total {g.sum():.3f}: where it comes from')
for i, t in enumerate(tot):
    ax[2].text(i, t, f'{t:.2f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()

* The tooth peaks just **after** each perfect square (n = k² + 1 needs a new row and column) and is 0 at k².
* The wasted area grows with n, but the score divides by √n, so each large-n puzzle is worth only 0.01 – 0.03.
* There are 900 of them, though: **101 – 1000 holds 21 of the 29 grid points.** A method that improves every large n a
  little beats hand-crafting the small ones.

## 3. How the scorer checks overlap

Two convex shapes are disjoint exactly when some line separates them. For two rectangles it is enough to test the four
edge directions (the *separating axis theorem*): project both squares onto each direction; if the two shadows don't
overlap on at least one of them, the squares don't overlap.

In [6]:
def shadows(ax, A, B, title):
    for q, col in [(A, '#4c7fb0'), (B, '#f4a63a')]:
        ax.add_patch(Polygon(corners(*q), fc=col, ec='#222', alpha=.85))
    best = None
    for k, ang in enumerate([A[2], A[2] + math.pi / 2, B[2], B[2] + math.pi / 2]):
        u = np.array([math.cos(ang), math.sin(ang)])
        pa = [np.dot(u, p) for p in corners(*A)]; pb = [np.dot(u, p) for p in corners(*B)]
        gap = max(min(pb) - max(pa), min(pa) - max(pb))
        off = -1.1 - .28 * k
        for pr, col in [(pa, '#4c7fb0'), (pb, '#f4a63a')]:
            ax.plot([min(pr), max(pr)], [off, off], color=col, lw=5, solid_capstyle='butt', alpha=.85)
        ax.text(3.05, off - .05, f'axis {k + 1}: ' + ('GAP' if gap > 0 else 'overlap'), fontsize=8,
                color='green' if gap > 0 else 'crimson')
    ax.set_title(title, fontsize=9); ax.set_xlim(-1, 4.3); ax.set_ylim(-2.4, 2.2); ax.set_aspect('equal'); ax.axis('off')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
shadows(ax[0], (0.6, 0.5, 0.0), (2.15, 0.9, math.pi / 4), 'separated: one axis shows a gap')
shadows(ax[1], (0.6, 0.5, 0.0), (1.55, 0.9, math.pi / 4), 'overlapping: every axis overlaps')
plt.suptitle('Shadows (projections) of both squares on the four edge directions (drawn below each pair, schematically)', y=1.0)
plt.tight_layout(); plt.show()

The official metric does this in floating point first. Any pair it cannot decide (gap or overlap smaller than 1e-8)
is re-checked in **interval arithmetic** at 50, 100, 200 and 500 digits, and passes only if it is **certified** separate.

So touching is allowed, but it has to be provable:

* the grid passes: 0.5, 1.5, angle 0 are exact in binary, the overlap interval is exactly [0, 0];
* the perfect n = 5 does not: 2 + √2/2 and π/4 cannot be written exactly, the rounded numbers overlap by ~1e-50.

In [7]:
def ev(sq, as_float64=False):
    f = (lambda v: repr(float(v))) if as_float64 else str
    rows = [dict(n=str(len(sq)), square_id=str(i), x=f(x), y=f(y), angle=f(a)) for i, (x, y, a) in enumerate(sq)]
    _, s, L, why = metric.evaluate_single_puzzle(len(sq), rows)
    return f'{s:.6f}  {why}'

print('grid n=5                         ', ev(build.grid(5)))
print('tilted n=5, exact contact        ', ev(build.five(gap=0)))
print('tilted n=5, gap 1e-12            ', ev(build.five()))

## 4. The float64 trap

Our first upload had the tilted n = 5 with a gap of **1e-20**, written with 50 digits. Locally it scored 28.8067.
On the leaderboard it scored **28.9377: exactly the grid.** The improvement was gone.

The reason: the scorer receives the CSV as a pandas DataFrame, and a CSV read with default settings turns every number
into a 64-bit float (about 16 significant digits). A gap of 1e-20 next to a coordinate of size 2 is below float64
resolution (2.2e-16 × 2), so it rounds away and the squares are back to touching, which cannot be certified.
Simulating that round trip shows it, and shows the fix:

> **Correction 2026-10-31: the board scores the float64 values of your CSV, not the text.** The scorer receives your file as a pandas DataFrame read with default settings, so a 45-digit coordinate reaches the metric as its nearest double. We had this backwards from 2026-10-29: four of our submissions certified 22.2605-22.2616 on their text and scored 22.6798, 25.1903, 22.6309 and 25.1809 on the board, and certifying the same files after a float64 round trip predicts all four to 7 decimals. So the 1e-12 gap below **is** required, and a file is only as good as its float64 round trip. Score locally with `pd.read_csv(path, float_precision='round_trip')`: correctly rounded float64 is what the board scores (six board scores match it exactly), while pandas' default parser is one ulp off on some long literals and mis-predicts by 0.1-0.2; when you merge other people's files, certify each packing after `repr(float(v))` and write that float text (the kit's `ensemble.py` does since 2026-10-31).
>
> What else changed in the 2026-10-29 metric (pull it with `kaggle kernels pull metric/squares-in-squares`; the kit's `metric.py` is that version): every square's box is padded by 1e-7 before the overlap test, coordinates are measured relative to square 0, |x| and |y| must be at most 10⁴ and |angle| at most 100, each number may have at most **100 characters** (it was 1000), and a side below √n − 1e-7 is rejected. The per-puzzle leaderboard was cleaned the same day: a new best now has to beat the old one by more than 1e-12 to show up.

In [8]:
import mpmath
rows = []
for g_ in ['1e-20', '1e-16', '1e-14', '1e-12', '1e-10']:
    sq = build.five(gap=mpmath.mpf(g_))
    rows.append((g_, ev(sq).split()[0], ev(sq, as_float64=True)))
pd.DataFrame(rows, columns=['gap', 'score, 50-digit strings', 'score after float64 round trip (what Kaggle sees)'])

**Rule: leave a gap of about 1e-12 at every contact** (or scale every packing about its centre by 1 + 1e-12).
It costs ~1e-12 per puzzle, nothing you will ever see on the leaderboard, and it survives float64.

## 5. A toy optimiser that rediscovers the tilt

No construction needed: give each square three numbers (x, y, angle), fix a box side L, and minimise a penalty

* how far each corner sticks out of the box, squared, plus
* for every pair, the smallest overlap of their shadows (section 3), squared.

Penalty 0 means a valid packing. Then bisect on L: shrink while some random start reaches penalty 0
(`opt_demo.py`, 40 lines, SciPy L-BFGS-B, 12 random starts per L).

In [9]:
rng = np.random.default_rng(0)
frames, lo, hi = [], math.sqrt(5), 3.0
t = time.time()
for it in range(9):
    Lt = (lo + hi) / 2
    r = opt_demo.fit(5, Lt, rng, starts=12)
    ok = r.fun < 1e-10
    frames.append((Lt, r.x.reshape(5, 3), ok, r.fun))
    hi, lo = (Lt, lo) if ok else (hi, Lt)
print(f'best feasible L found: {hi:.4f} (optimum 2 + sqrt(2)/2 = {L5:.4f}) in {time.time() - t:.0f}s')
fig, ax = plt.subplots(1, 5, figsize=(16, 3.6))
for a, (Lt, P, ok, pen) in zip(ax, frames[:4] + [max((f for f in frames if f[2]), key=lambda f: -f[0])]):
    for x, y, an in P:
        a.add_patch(Polygon(corners(x, y, an), fc='#f4a63a' if abs(math.sin(2 * an)) > .05 else '#4c7fb0', ec='#222', alpha=.85))
    a.add_patch(Rectangle((0, 0), Lt, Lt, fill=False, ec='crimson', lw=2))
    a.set_title(f'box {Lt:.3f}: ' + ('valid' if ok else f'overlap {pen:.1e}'), fontsize=9)
    a.set_xlim(-.3, 3.2); a.set_ylim(-.3, 3.2); a.set_aspect('equal'); a.axis('off')
plt.suptitle('bisection on the box side; the last panel is the smallest valid box found', y=1.02)
plt.tight_layout(); plt.show()

It finds a tilted middle square on its own, but stops a little above the optimum: the last few thousandths need the
squares to sit *exactly* in contact, which a penalty method approaches only slowly. Real solvers add better moves
(simulated annealing, swaps, symmetry) and a final exact "polish" step.

## 6. A family that works for hundreds of n: staircases + a tilted block

Göbel's idea scales up. Fill each corner of the box with a **staircase** of upright squares (t rows hold
t(t+1)/2 squares). The outer corners of a staircase lie on a 45° line. Between the four staircases, along the
diagonal, put a **w × h block of squares turned 45°** resting on those lines.
The box side is then `s = max((TL+BR+2)/2 + w/√2, (TR+BL+2)/2 + h/√2)`.

David Ellsworth's enumerator (`enum-45deg-rect-packings.cpp`, github.com/Davidebyzero/packing_squares_in_squares__tools)
searches all staircase sizes and block shapes up to n = 2,117 and prints the side and the shape, e.g.
`s(739) = 27.6924: 13 x 25: 17,(1),9,(0),18,(0),9,(1)`. It prints no coordinates; `gobel.py` builds them from the shape.

In [10]:
ENUM = next(p for p in [f'{SRC}/enum45.txt', '../scratch/enum45.txt'] if os.path.exists(p))
import gobel
fam = gobel.family(ENUM)
fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
for a, n_ in zip(ax, [17, 26, 739]):
    s_, sq_ = fam[n_]
    draw(a, gobel.dilate(sq_, s_), f'45-degree family, grid would need {math.ceil(math.sqrt(n_))}')
plt.tight_layout(); plt.show()
print(f'{len(fam)} values of n get a 45-degree packing from the enumerator shapes and their L extensions (section 8)')

## 7. Standing on shoulders: the best-known table

For every n ≤ 324 (and some larger n) the best packings known are collected by David Ellsworth
(kingbird.myphotos.cc/packing/squares_in_squares.html, building on Erich Friedman's survey and decades of work).
Many are **not** 45°: n = 17 tilts a band of squares by about 39.8°.
We use the coordinates as mirrored in github.com/evand/square-packing (`site/www/data/p/*.json`, centres in float64,
angles in degrees), scale each packing by 1 + 1e-12 about its centre, and keep it only if the official metric certifies
it after a float64 round trip: all 350 pass. The packings are Ellsworth's; credit to him and to everyone in that table.

In [11]:
REC_PATH = next(p for p in [f'{SRC}/records.json', '../out/records.json'] if os.path.exists(p))
import json
REC = {int(k): v for k, v in json.load(open(REC_PATH)).items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
for a, n_ in zip(ax, [17, 41, 300]):
    draw(a, REC[n_]['squares'], f'best known n={n_}')
plt.tight_layout(); plt.show()
print(len(REC), 'certified record packings; they beat the grid for', sum(r['L'] < math.ceil(math.sqrt(k)) - 1e-9 for k, r in REC.items()))

## 8. Two closures that are free

* **Add an L.** A packing of n squares in side S, plus a column of ⌊S⌋+1 and a row of ⌊S⌋ upright squares along two
  edges, packs n + 2⌊S⌋ + 1 squares in side S + 1. So every good packing also improves a whole chain of larger n.
* **Remove squares.** A packing of m squares minus any m − n of them packs n squares in a box no larger.
  So if n = 740 has a good packing, n = 739, 738, … can borrow it until something better turns up.

In [12]:
import make_sub
side10, sq10 = make_sub.add_L_float(make_sub.floats(*fam[5]))
fig, ax = plt.subplots(1, 2, figsize=(10, 4.8))
draw(ax[0], make_sub.floats(*fam[5]), 'n=5')
draw(ax[1], sq10, 'n=5 + an L of 2+2+1 squares = n=10')
plt.tight_layout(); plt.show()

## 9. The submission

Best per n from the grid, the 45° family, the records, then both closures. Each step's total, measured from the sides:

In [13]:
stages = []
t = time.time()
best = make_sub.build(ENUM, REC_PATH, stages)
rows = make_sub.write(best, f'{OUT}/submission.csv')
print(f'{rows} rows written to {OUT}/submission.csv in {time.time() - t:.0f}s')
fig, ax = plt.subplots(figsize=(10, 3.6))
names, vals = zip(*stages)
ax.bar(names, vals, color=['#999'] + ['#4c7fb0'] * (len(vals) - 1))
for i, v in enumerate(vals):
    ax.text(i, v + .1, f'{v:.3f}', ha='center')
ax.set_ylim(20, 30); ax.set_ylabel('total score (lower is better)'); ax.set_title('what each idea is worth')
plt.xticks(rotation=12); plt.tight_layout(); plt.show()

Certify a sample with the official metric on the file's text, as the board reads it (the whole file takes a few minutes;
our upload of this exact file scored on the leaderboard what it scores locally).

In [14]:
s = pd.read_csv(f'{OUT}/submission.csv', float_precision='round_trip')   # float64, correctly rounded, as the board reads it
pick = [5, 17, 41, 300, 331, 739, 1000]
part = s[s.n.astype(int).isin(pick)]
sol = pd.DataFrame({'id': range(len(pick)), 'n': pick})
t = time.time(); got = metric.score(sol, part, 'id')
print(f'puzzles {pick}: official {got:.6f}  vs  from the sides {sum(best[k][0] / math.sqrt(k) - 1 for k in pick):.6f}  ({time.time() - t:.0f}s)')

## 10. Where to go next: what came after this notebook

This notebook stops at 22.72. Since then the board has moved by about 0.47, and almost all of it came from four ideas.
Each has a public notebook that explains it properly; this section is the map.

In [15]:
steps = [('grid', 28.94), ('this notebook', 22.72), ('certified merge', 22.598), ('fault-line DP\n+ basin hopping', 22.375),
         ('+ gadget quench', 22.372), ('merge of new\npublic files\n(2026-10-24)', 22.308), ('board top\n(2026-10-24)', 22.225)]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.3, 1]})
x = np.arange(len(steps)); y = [v for _, v in steps]
ax.plot(x[1:], y[1:], 'o-', color='tab:blue')
for k, (lab, v) in enumerate(steps[1:], 1):
    ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.set_xticks(x[1:]); ax.set_xticklabels([l for l, _ in steps[1:]], fontsize=8)
ax.set_ylabel('score (lower is better)'); ax.set_title('After 22.72 (the grid, 28.94, is off the chart)')
ax.grid(alpha=.3)
# fault line exactly as the DP composes it (from the Fault-line notebook): three 2 x 2 gadgets, K = 6, overhang f
K, F0, gad = 6, 0.55, [(2, 2), (2, 2), (2, 2)]
lo, hi, ox, oy, boxes = [K] * K, [K] * K, 0, K, []
for w, h in gad:
    oy -= h; lo[ox:ox + w] = [oy] * w; hi[ox:ox + w] = [oy + h] * w; boxes.append((ox, oy, w, h)); ox += w
for i in range(K):
    for j in range(lo[i]):
        bx.add_patch(Rectangle((i, j), 1, 1, fc='#4c7fb0', ec='#222', lw=.4))
    for j in range(hi[i], K):
        bx.add_patch(Rectangle((i + F0, j + F0), 1, 1, fc='#7fb07f', ec='#222', lw=.4))
for k, (gx, gy, w, h) in enumerate(boxes):
    bx.add_patch(Rectangle((gx, gy), w + F0, h + F0, fc='#f4a63a', ec='#222', lw=.8, alpha=.85))
    bx.text(gx + (w + F0) / 2, gy + (h + F0) / 2, f'gadget {k + 1}\n{w}x{h}+dm', ha='center', va='center', fontsize=8)
bx.add_patch(Rectangle((0, 0), K + F0, K + F0, fill=False, ec='crimson', lw=2))
bx.set_xlim(-.3, K + F0 + .3); bx.set_ylim(-.3, K + F0 + .3); bx.set_aspect('equal'); bx.axis('off')
bx.set_title('fault line: blue lattice, green lattice shifted by f,\ngadgets (w+f) x (h+f) along the seam; box K + f', fontsize=9)
plt.tight_layout(); plt.show()

| idea | what it does | whose | explained in |
|---|---|---|---|
| **certified merge** | take the best packing per n from every public file, re-check each with the official metric after a float64 round trip, keep the winners | **ours** (rank 1 on 2026-10-21 at 22.5978) | [Certified Merge of Public Packings](https://www.kaggle.com/code/alexchilton/certified-merge-of-public-packings-lb-22-5978) |
| **fault-line DP** | for n = K² + d, keep two K × K grids and shift one by a small f; a dynamic programme chooses which small dense "gadgets" sit on the seam to hold the d extra squares, so the side is K + f | idea: [Chris Deotte](https://www.kaggle.com/cdeotte) (his notebook ships the packings); **the public code is ours** (`fault_dp.py`, every composition certified) | [Fault-line DP + Basin Hopping](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620) |
| **gadget quench** | a gadget built from an exact construction (overhang f = (p + q√2)/r) is rigid; every other gadget has slack, so loosen its box, kick it and squeeze it again with a smooth solver, then rerun the DP with the smaller gadgets | **ours** (−0.0026 on 78 gadgets) | [Fault-line DP](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620), section 4b |
| **basin hopping / LP snap** | kick a packing, re-optimise, keep improvements; a linear programme snaps contacts exactly | many; Kirill's LP snap | [Fault-line DP](https://www.kaggle.com/code/alexchilton/fault-line-dp-basin-hopping-lb-22-2620); [Kirill's EDA + Snap LP](https://www.kaggle.com/code/kirilltym/22-3720-cpu-cuda-mps-eda-snap-lp) |
| **GPU packing search, seam DP** | massively parallel local search; D4-symmetric seam variants | lightningv08; Janus | [CUDA Packing Search](https://www.kaggle.com/code/lightningv08/squares-in-squares-cuda-packing-search), [D4 Seam DP](https://www.kaggle.com/code/jiwasawa/squares-in-squares-d4-seam-dp-dual-gpu-lbfgs) |
| **ensembles** | merge many public files per n | Chris Deotte and others | [Ensemble](https://www.kaggle.com/code/cdeotte/squares-in-squares-ensemble-22-376) |

The advice from the first version still holds:

1. **Always** score locally before you upload. Read your file as correctly rounded float64 (`pd.read_csv(path, float_precision='round_trip')`), as the board does; pandas' default parser is not correctly rounded and mis-predicts; we wrongly said `dtype=str` here from 2026-10-29 to 2026-10-31 (section 4).
2. **n > 324 is where the points are**: the table stops at 324, and the fault-line DP is the first method that
   reaches every non-square n above it.
3. **Merging public files** (lessons from 2026-10-29): certify each packing after a float64 round trip and write the float text (45-digit text that certifies only at full precision scores as overlapping on the board); skip every puzzle outside n = 1…1000 (one public file carries puzzles up to n = 3427, and a merge that indexes by n crashes on them); and re-pull the official metric whenever the organisers announce a change, because the old copy keeps passing files the board now treats differently.
4. The top of the board is about 0.08 below the best public merge (22.2248 vs 22.3084 on 2026-10-24), so the next idea is not public yet. Section 11 lists what we measured does **not** close it.

Milestone prizes go to the leader on 2027-01-17 and 2027-04-18, and **ties go to whoever submitted first**.

## 11. What does not close the gap (measured 2026-10-24)

The gadget quench of section 10 found its gains in one pass. We then asked whether more of the same helps. It does not:

| arm | what it changes | result |
|---|---|---|
| longer squeeze | 300 s per gadget instead of 60 s, 6 top-leverage keys (18×19+7, 20×21+10, 8×11+2, 5×17+11, 7×17+11, 7×16+10) | **the same f at all 6 keys**; one new gadget worth 10⁻⁶ |
| other starting points | start each key from its neighbour key's gadget (one more row, column or square), cropped back | **0 of 14** runs beat the key's own gadget (e.g. 8×11+2: 0.59138 own, 0.64724 from the taller neighbour) |
| whole-puzzle relaxation | float64 L-BFGS on every square of a puzzle, then certify | 90 puzzles, total gain 3 × 10⁻⁶ |
| two fault lines | split the extra squares over two seams | one seam already costs f ≈ 0.52 at K = 25, so two cost ≥ 1.04, more than K + 1; 0 of about 400 puzzles qualify |

So the squeeze is a **one-shot harvest**. What is left are the gadgets no squeeze can move: the four biggest levers in the
DP (8×8+1, 2×2+1, 12×12+4, 16×16+9) are exact 45° constructions, rigid at a local optimum (overhangs like √2/2 and
(1+√2)/…). Beating them needs a **different construction** at the same footprint, not a better optimiser.
That is our best guess for where the leaders' 0.08 sits, and it is a guess: no public file contains such a gadget.

*Credits: Göbel (1979) for the 45° idea; David Ellsworth for the enumerator and the best-known table;
Erich Friedman's survey; github.com/evand/square-packing for machine-readable coordinates; the authors of the notebooks
linked above. If this helped, an upvote is appreciated.*

## 11. Bonus: our current best, certified

Sections 1 to 9 build 22.72 from first principles. The starter-kit dataset also carries `best_current.csv`: our current best file.
It is the per-puzzle best, certified, of our own packings (the fault-line DP with basin hopping, the gadget quench, and a few
packings found by an LLM agent, Gemini, iterating against the official metric) and the public outputs of
**kmmmvs** (*Squares Dashboard Public Batch 20261025*, 87 puzzles), **aqibrazadev** (*Top LB Blending + Visualization*, 42),
**kirilltym** (*CPU/CUDA/MPS EDA+ProjUnion*, 39) and **avazaku** (*Square Packing with Tilt Engine*, 13); we also merged
seshurajup, prvsiyan, lightningv08 and nilanjansaha123. The 2026-10-26 re-merge (certified 22.298226) took 48 puzzles from kmmmvs,
31 from aqibrazadev, 4 from **nilanjansaha123** (*Certified Portfolio Merge + Projection Quench*) and 2 from prvsiyan. A second 2026-10-26 re-merge (certified 22.298213) took 5 puzzles from [Ridham Kavani, *Neighbour Transfer + Squeeze*](https://www.kaggle.com/code/ridhamkavani/squares-in-squares-neighbour-transfer-squeeze) (a fork of Malyshev Danil's notebook). A third 2026-10-26 re-merge (certified 22.298157) took 16 puzzles from [prvsiyan, *Auto-updater*](https://www.kaggle.com/code/prvsiyan/auto-updater), 2 from [nilanjansaha123, *Certified Portfolio Merge + Proj Quench*](https://www.kaggle.com/code/nilanjansaha123/s-in-s-certified-portfolio-merge-proj-quench) and 1 from [zkskhurram, *Kaggle Squares in Squares*](https://www.kaggle.com/code/zkskhurram/kaggle-squares-in-squares). All credit for their packings goes to them.
The ideas behind all of it and their authors are in the table of section 10.

The cell below does not trust the file. For every n it re-checks both candidates, this notebook's build and `best_current.csv`,
with the official metric after a float64 round trip (what the board scores), keeps the smaller valid side and writes it as float text. It takes a few minutes.

The 2026-10-28 re-merge (certified 22.266616) took 513 puzzles from [busyaprime, *Public Packings + fq Quench*](https://www.kaggle.com/code/busyaprime/33-lb-22-2683156-public-packings-fq-quench) and 1 from malyshevdanil's *Neighbour Transfer + Squeeze*. All credit for those packings goes to them.
The 2026-10-31 re-merge (certified 22.260344, float64-certified as the board reads it) took 280 puzzles from [lucifer19, *Think Select Reclose V24: The Frontier Moves*](https://www.kaggle.com/code/lucifer19/think-select-reclose-v24-the-frontier-moves), 5 from [sigmaborov, *Certified Contact Polish*](https://www.kaggle.com/code/sigmaborov/squares-in-squares-certified-contact-polish) and 3 from [malyshevdanil, *Living Atlas of Tight Packings*](https://www.kaggle.com/code/malyshevdanil/squares-in-squares-living-atlas-of-tight-packings).


In [ ]:
import ensemble
BEST = next(p for p in [f'{SRC}/best_current.csv', '../submissions/best_local.csv'] if os.path.exists(p))
mine, cur = pd.read_csv(f'{OUT}/submission.csv', dtype=str), pd.read_csv(BEST, dtype=str)
arr = lambda g: g.sort_values('square_id', key=lambda s: s.astype(int))[['x', 'y', 'angle']]
def cert(n, g):
    # float64 only: the board reads the CSV as float64 (2026-10-31: four submissions scored their float64 value to 7 decimals)
    return ensemble.certify(n, arr(g).to_numpy(float))
rows = lambda n, r: pd.DataFrame({'n': n, 'square_id': range(n), 'x': [t[0] for t in r[2]], 'y': [t[1] for t in r[2]], 'angle': [t[2] for t in r[2]]})
t, keep, total, took = time.time(), [], 0.0, 0
for n in range(1, 1001):
    a, b = cert(n, mine[mine.n.astype(int) == n]), cert(n, cur[cur.n.astype(int) == n])
    use_b = b is not None and (a is None or b[0] < a[0])
    side_n = (b if use_b else a)[0]
    took += use_b; total += side_n / math.sqrt(n) - 1
    keep.append(rows(n, b if use_b else a))                 # written as float text, exactly what the board scores
out = pd.concat(keep)
out['id'] = range(len(out))
out[['id', 'n', 'square_id', 'x', 'y', 'angle']].to_csv(f'{OUT}/submission.csv', index=False)
print(f'best_current wins {took} of 1000 puzzles; certified total {total:.6f} ({time.time() - t:.0f}s)')

## 12. What we tried that did not work (2026-10-26 to 2026-10-27)

A focused hunt through outside repos, papers and the Kaggle discussion board turned up several ideas.
None of them beat what the fault-line DP and gadget quench already found.
Only measured results are in this table.

| idea | what we measured | result |
|---|---|---|
| Whole-puzzle float64 L-BFGS relax | Re-optimised every square of 90 puzzles, then certified with the official metric. | Gain 3×10⁻⁶ in total. Next to nothing. |
| Ellsworth catalogue variants as gadget donors | Cut 317 non-record catalogue packings into 10,606 gadgets, kept the 1,430 not already in our bank, reran the fault-line DP over n = 325–1000 with them added. | 0 puzzles improved. |
| Dostashev's published records (12 values of n) | Compared our best_local side to all 12 (n = 129, 155, 156, 175, 176, 204, 211, 232, 241, 264, 303, 304). | We equal or beat every one (for example n = 303: 17.906978 vs 17.909958). |
| Junction V-merge (soft-melt two chain gadgets into one band) | Ran a 60 s soft quench on the two-gadget puzzles 689, 742, 856 and 546; 915 was skipped because 856, a smaller puzzle, had already not finished. | No run finished at n ≥ 689. The one puzzle that did finish, 546, came out **worse** by 0.138. |
| Two-band tilted-stick frame | Built the frame at K = 22, f = 0.79903 (n = 498) over band widths h = 3 to 8. | Best count 491 of 498 squares. 7 short of the target. |
| Longer gadget optimisation (1200–1500 s per key) | Ran 21 blocking keys through the optimiser for 1200–1500 s each, across three rounds (2026-10-26, 2026-10-27). | 14 of the 21 keys never moved off the bank f. |
| Asymptotic wasted-space constructions | Compared Ryu's R1' (c ≈ 5.657√k) and related asymptotic bounds against Fang's c = k − 1. | Ryu loses to Fang at every k ≤ 37. No explicit constant from these papers helps below k ≈ 32. |
| Certifying on the file's decimal text (2026-10-29 to 10-31) | We believed the 2026-10 metric update made the board read the submitted text, and certified and wrote 45-digit text. The board reads the CSV as float64: four submissions certified 22.2605-22.2616 as text and scored 22.6798, 25.1903, 22.6309 and 25.1809; the float64 round trip predicts all four to 7 decimals. Fixed 2026-10-31: certify after float64, write float text. | The file that 'proved' text-reading certified the same both ways. Test a claim with a file where the two readings disagree. |
| Our own merge, after the 2026-10 metric update (found 2026-10-29) | It read every public file as float64. Public bests now carry 45-digit coordinates that certify only as text: busyaprime's file scores 22.261999860 as text, 22.26244 after float64. The merge rejected every one of them for about a day and reported no error. That fix was wrong (the row above): the board reads float64, so text-only packings score as overlapping. The same day a public file with puzzles up to n = 3,427 crashed the merge; puzzles outside 1 … 1000 are now skipped. | A silent failure, not an idea that lost. If public scores beat your merge, score their file both ways before blaming timing. |
| This notebook's own bonus merge (section 11), same bug (found 2026-10-29 evening) | Section 11 still read `best_current.csv` as float64, so 387 of its 1,000 packings failed as overlapping and fell back to this notebook's build: versions 29 and 30 certified 22.631322 against the kit's 22.261637. Fixed: section 11 certifies the text and writes it unchanged. | Fix a bug everywhere the same read happens, not only where it was found. |

**Open, not measured:** the L-shaped frame idea (chains of gadgets along one straight arm) was ruled out by a geometry argument, not by a run.
A straight arm pays the overhang cost once per gadget, not once for the whole arm, so it collapses back into the diagonal chain search we already do.

We do not use the metric exploits raised on the Kaggle discussion board: large angles and 10⁶⁴-scale coordinate shifts that make the broad-phase overlap check disagree with the certified score, which is how a few teams post negative scores.
Our own numbers here are all certified with the official metric (after a float64 round trip, as the board reads the file), the strict gate, not the exploited one.

**Takeaways:**
1. Every arm tried this round is a null or a rounding error. The easy gains from squeezing existing gadgets are gone.
2. The leaders' edge looks like better gadgets at the same deficit, not a different geometry, donor source, or published record we missed.
3. None of this changes what we ship: best_local stays the strict, exploit-free certification.


---
If you used `best_current.csv` or any file of the [starter kit](https://www.kaggle.com/datasets/alexchilton/squares-in-squares-starter-kit), an upvote on the dataset is the only thing that keeps it updated with each new merge. Thanks.